# Fase4: auditoría y resultados exploratorios

Ejecutar después de auditor.py --preflight, --pruebas, analizar.py, complementar.py y documentar.py. Entradas: guardados etapa2, auditorías y tablas etapa3. Python3.12.14/sklearn1.9.1/54lock/PYTHONDONTWRITEBYTECODE=1/semilla20261010. Celdas solo lectura y recálculo; no escriben resultados ni cargan modelos. Generador ejecuta realmente con exec secuencial; salidas de esta ejecución.

Primario global fijo,165pares/igualpeso/meta5%. Advertencias completas en INFORME_FASE4.md. Resultados de políticas individuales exploratorios; ningún experimento nuevo autorizado.

In [1]:
from pathlib import Path
import sys, os, json, math, csv, platform, importlib.metadata as md, subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experimentos_memoria_comun_objetivo3_v1/etapa3/auditor.py').exists())
stage = root/'experimentos_memoria_comun_objetivo3_v1/etapa3'
sys.path.insert(0,str(stage))
from auditor import load, rows, metrics, select, CAND
lock = dict(line.strip().split('==') for line in (root/'experimentos_objetivo3_v5/etapa1/requirements.lock.txt').read_text().splitlines() if '==' in line and not line.startswith('#'))
assert platform.python_version()=='3.12.14' and len(lock)==54
assert all(md.version(k)==v for k,v in lock.items()) and md.version('scikit-learn')=='1.9.1'
assert os.environ.get('PYTHONDONTWRITEBYTECODE')=='1'
assert subprocess.run([sys.executable,'-m','pip','check'],capture_output=True).returncode==0
seed=20261010
assert load(stage/'AUDITORIA_INDEPENDIENTE.json')['aprobado'] and load(stage/'AUDITORIA_COMPLEMENTARIA.json')['aprobado']
print('Entorno exacto; semilla contractual',seed,'; cero fits/cargas de modelos. Entradas: etapa2 guardada y tablas etapa3. Lectura y cálculo sin escritura.')

Entorno exacto; semilla contractual 20261010 ; cero fits/cargas de modelos. Entradas: etapa2 guardada y tablas etapa3. Lectura y cálculo sin escritura.


In [2]:
external=rows(root/'experimentos_memoria_comun_objetivo3_v1/etapa2/resultados/predicciones_externas.csv')
recalculated={p:metrics([r for r in external if r['policy']==p]) for p in CAND+['global']}
base=recalculated['persistencia']['MAE']
assert base>0 and math.isfinite(base)
for p,m in recalculated.items():
    m['improvement_percent']=100*(base-m['MAE'])/base
    print(p, json.dumps(m))
print('Primario global: meta5% satisfecha =',recalculated['global']['improvement_percent']>=5)
assert recalculated['global']['MAE']==load(stage/'AUDITORIA_INDEPENDIENTE.json')['global']['MAE']

persistencia {"n": 165, "MAE": 3.429267191379531, "RMSE": 4.314920647368904, "sesgo": -2.0960105973100562, "improvement_percent": 0.0}
memoria {"n": 165, "MAE": 3.004299017227967, "RMSE": 4.212665669505837, "sesgo": -0.965693505233788, "improvement_percent": 12.392390281511053}
comun {"n": 165, "MAE": 3.436546823383837, "RMSE": 4.476778255626107, "sesgo": -1.4260966275450535, "improvement_percent": -0.21227952206830963}
comun_correccion {"n": 165, "MAE": 3.436546823383837, "RMSE": 4.476778255626107, "sesgo": -1.4260966275450535, "improvement_percent": -0.21227952206830963}
global {"n": 165, "MAE": 3.3676098407589707, "RMSE": 4.306243910848592, "sesgo": -1.5242014817652496, "improvement_percent": 1.7979745286559843}
Primario global: meta5% satisfecha = False


In [3]:
coefficients=rows(stage/'tablas/coeficientes_todos_41.csv')
mem=[r for r in coefficients if r['component']=='memoria']
corr=[r for r in coefficients if r['component']=='correccion']
assert len(mem)==11 and all(float(r['coef_lag_standardized'])==0 for r in mem)
assert len(corr)==10 and all(float(r['coef_lag_standardized'])==float(r['intercept_points'])==0 for r in corr)
assert all(r['intercept_in_median_interval']=='True' for r in coefficients)
print('Memoria:11ajustes coeficiente0; predicción=previo+constante histórica. No demuestra dinámica ni causa exclusiva alpha1.')
for r in mem:
    if r['role']=='externa': print(r['evaluated'],r['n_train'],r['intercept_points'])
print('Corrección:10ajustes coeficiente/intercepto0. Común/común+corrección iguales en165externas, bajo configuración actual.')

Memoria:11ajustes coeficiente0; predicción=previo+constante histórica. No demuestra dinámica ni causa exclusiva alpha1.
2022 33 -1.2006135943734648
2023 66 1.6516505643865287
2024 99 2.341502912644245
2025 132 2.8590455777240322
2025 132 2.8590455777240322
Corrección:10ajustes coeficiente/intercepto0. Común/común+corrección iguales en165externas, bajo configuración actual.


In [4]:
internal=rows(root/'experimentos_memoria_comun_objetivo3_v1/etapa2/resultados/predicciones_internas_originales.csv')
for year in range(2022,2026):
    ms={c:metrics([r for r in internal if int(r['external'])==year and r['candidate']==c]) for c in CAND}
    winner,p1,p2=select(ms)
    print(year,'ganador',winner,'poolMAE',p1,'poolRMSE',p2)
print('2021 externo fijo;2021 internas respaldadas y puntuadas. Solo internas seleccionan; externas memoria no rescatan primario.')

2022 ganador persistencia poolMAE ['persistencia', 'memoria', 'comun', 'comun_correccion'] poolRMSE ['persistencia', 'memoria', 'comun', 'comun_correccion']
2023 ganador persistencia poolMAE ['persistencia'] poolRMSE ['persistencia']
2024 ganador persistencia poolMAE ['persistencia', 'memoria'] poolRMSE ['persistencia']
2025 ganador memoria poolMAE ['memoria'] poolRMSE ['memoria']
2021 externo fijo;2021 internas respaldadas y puntuadas. Solo internas seleccionan; externas memoria no rescatan primario.


In [5]:
annual=rows(stage/'tablas/por_anio.csv')
territorial=rows(stage/'tablas/por_territorio.csv')
for r in annual:
    if r['policy']=='global': print(json.dumps(r,ensure_ascii=False))
t=[r for r in territorial if r['policy']=='global']
print('Territorios mejoran/empeoran:',sum(float(r['benefit_MAE_points'])>0 for r in t),sum(float(r['benefit_MAE_points'])<0 for r in t))
assert all(float(r['benefit_MAE_points'])==0 for r in annual if r['policy']=='global' and int(r['year'])<2025)
print('Toda mejora global2025.165pares dependientes; no165observaciones independientes. Tablas completas en tablas/.')

{"year": "2021", "policy": "global", "n": "33", "MAE_points": "2.6402236932057406", "RMSE_points": "3.5097490978755155", "bias_points": "1.4070902157455416", "benefit_MAE_points": "0.0", "improve_pairs": "0", "worsen_pairs": "0", "tie_pairs": "33", "direction_match_pairs": "0"}
{"year": "2022", "policy": "global", "n": "33", "MAE_points": "5.091966313567663", "RMSE_points": "5.553520055345682", "bias_points": "-4.506232893548528", "benefit_MAE_points": "0.0", "improve_pairs": "0", "worsen_pairs": "0", "tie_pairs": "33", "direction_match_pairs": "0"}
{"year": "2023", "policy": "global", "n": "33", "MAE_points": "3.2072717302598033", "RMSE_points": "4.010835597004481", "bias_points": "-2.250915534656569", "benefit_MAE_points": "0.0", "improve_pairs": "0", "worsen_pairs": "0", "tie_pairs": "33", "direction_match_pairs": "0"}
{"year": "2024", "policy": "global", "n": "33", "MAE_points": "3.9602791908251427", "RMSE_points": "4.921069321485291", "bias_points": "-3.63506241817535", "benefit_M

In [6]:
audit=load(stage/'AUDITORIA_INDEPENDIENTE.json')
print('Cobertura',audit['coverage'],'fits históricos',audit['fits_historicos'],'consumo histórico',audit['ledger_used_seconds'])
print('Nuevos fits0; modelos cargados0. Todos2021–2025vistos:exploratorio, sin confirmación independiente/causalidad.')
print('Advertencias:procedencia local no autenticada,Guainía,IPMhogares-alumnos,colegio-residencia,ecología,dependencia,pocos años.')
print('PROPUESTAS_FUTURAS.md: referencia constante y contraste de rezago/regularización; requieren protocolo/autorización separados. Ningún experimento implementado.')

Cobertura {'internal': 1320, 'records': 16, 'external': 825, 'unique': 528, 'lineage': 1320, 'global_checks': 5} fits históricos {'externa': 16, 'interna': 24, 'repeticion': 1} consumo histórico 58.63946153299912
Nuevos fits0; modelos cargados0. Todos2021–2025vistos:exploratorio, sin confirmación independiente/causalidad.
Advertencias:procedencia local no autenticada,Guainía,IPMhogares-alumnos,colegio-residencia,ecología,dependencia,pocos años.
PROPUESTAS_FUTURAS.md: referencia constante y contraste de rezago/regularización; requieren protocolo/autorización separados. Ningún experimento implementado.


## Gráficos guardados (puntos y n explícitos)

![MAE por año](graficos/mae_por_anio.png)

![Beneficio territorial](graficos/beneficio_territorial.png)

![Nivel y rezago](graficos/memoria_nivel_y_rezago.png)

Son recálculos de guardados, sin fits. No confirmación independiente, causalidad ni avance automático.